# 01 — Market Data Sanity

Tujuan: benar-benar melihat panel market IDX-Trade sendiri sebelum menyentuh model.

Notebook ini read-only. Isi `MARKET_PANEL` dengan path parquet/CSV historis yang aman untuk development.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

MARKET_PANEL = Path(r"CHANGE_ME")

In [ ]:
if not MARKET_PANEL.exists():
    raise FileNotFoundError(
        "Set MARKET_PANEL ke file historis lokal. Jangan arahkan ke protected forward outcomes."
    )

df = pd.read_parquet(MARKET_PANEL) if MARKET_PANEL.suffix.lower() == ".parquet" else pd.read_csv(MARKET_PANEL)
print("shape:", df.shape)
display(df.head())

## Struktur dasar

In [ ]:
print("columns:", len(df.columns))
display(pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_rate": df.isna().mean(),
}).sort_values("missing_rate", ascending=False).head(30))

if "ticker" in df:
    print("tickers:", df["ticker"].nunique())
if "date" in df:
    dates = pd.to_datetime(df["date"], errors="coerce")
    print("date range:", dates.min(), "->", dates.max())

## Duplicate / identity sanity

In [ ]:
identity = [c for c in ("ticker", "date") if c in df.columns]
if len(identity) == 2:
    dup = df.duplicated(identity, keep=False)
    print("duplicate ticker/date rows:", int(dup.sum()))
    if dup.any():
        display(df.loc[dup].sort_values(identity).head(20))

## Lihat satu ticker through time

In [ ]:
TICKER = "BBCA"

one = df[df["ticker"].astype(str).str.upper().eq(TICKER)].copy()
if "date" in one:
    one["date"] = pd.to_datetime(one["date"])
    one = one.sort_values("date")
print(TICKER, "rows:", len(one))
display(one.tail(10))

In [ ]:
price_candidates = [c for c in ("close", "Close", "regular_close", "ClosePrice") if c in one.columns]
if price_candidates and len(one):
    col = price_candidates[0]
    ax = one.plot(x="date", y=col, figsize=(12, 4), title=f"{TICKER} — {col}")
    ax.set_ylabel(col)
    plt.show()
else:
    print("No obvious close column found; inspect df.columns and choose the canonical close field manually.")

## Pertanyaan yang harus bisa kamu jawab setelah notebook ini

1. Panel ini satu row merepresentasikan apa?
2. Berapa ticker dan rentang waktunya?
3. Apakah ada duplicate ticker/date?
4. Kolom harga mana yang canonical/raw/adjusted?
5. Missingness terbesar datang dari field apa dan apakah itu expected?